# Tutorial 2 — Zadeh's two-doctors paradox

Zadeh (1986) showed that **Dempster's rule of combination can produce dramatically counter-intuitive results** when two sources strongly disagree. This notebook walks through the canonical example, then shows how the alternative rules shipped in `carla-evidence` v0.2.0 — Yager, Dubois-Prade, PCR5, and PCR6 — handle the same conflict more sensibly.

The lesson is methodological: there is no "best" combination rule. Each makes a different commitment about *what to do with conflicting mass*. You pick the rule whose commitment matches your application's risk profile.

**Reference**: Zadeh, L. A. (1986). A simple view of the Dempster-Shafer theory of evidence and its implication for the rule of combination. *AI Magazine*, 7(2), 85–90.

In [ ]:
from carla_evidence import Frame, MassFunction
from carla_evidence.combination import (
    conjunctive, dempster, disjunctive, yager, dubois_prade, pcr5, pcr6, mean,
)

## The setup

A patient is examined by two physicians. The diagnostic frame is

$$\Theta = \{\text{meningitis},\ \text{contusion},\ \text{brain-tumor}\}.$$

Doctor 1 strongly suspects meningitis but allows a 1% chance of brain tumour. Doctor 2 strongly suspects a contusion but also allows 1% for brain tumour.

In [ ]:
theta = Frame.of("meningitis", "contusion", "brain_tumor")

doctor_1 = MassFunction(theta, {("meningitis",): 0.99, ("brain_tumor",): 0.01})
doctor_2 = MassFunction(theta, {("contusion",): 0.99, ("brain_tumor",): 0.01})

print("Doctor 1:", doctor_1)
print("Doctor 2:", doctor_2)

## The Dempster paradox

Dempster's rule normalises by ``1 / (1 - K)``. The conflict mass ``K`` is enormous (`0.99·0.99 + 0.99·0.01 + 0.01·0.99 = 0.9999`), so the only surviving (non-conflicting) product — the tiny `0.01 · 0.01 = 0.0001` on `brain_tumor` — gets divided by `0.0001` and **becomes certainty**.

In [ ]:
out_dempster = dempster(doctor_1, doctor_2)
print("Dempster:", out_dempster)
print(f"  m(brain_tumor) = {out_dempster.mass(('brain_tumor',))}")

Both doctors gave only **1%** to the tumour hypothesis, yet Dempster reports it as a **dead certainty**. This is the paradox: a rule that takes the conflict as an *informational* signal collapses any survivor of total disagreement into total commitment.

## Yager: dump conflict on $\Theta$

Yager's rule keeps the conjunctive product, but instead of normalising the conflict away, it sends the conflicting mass to $\Theta$ — admitting *we don't know* rather than artificially boosting one survivor.

In [ ]:
out_yager = yager(doctor_1, doctor_2)
print("Yager:", out_yager)
print(f"  m(brain_tumor) = {out_yager.mass(('brain_tumor',)):.6f}")
print(f"  m(Theta)       = {out_yager.mass(theta.omega):.6f}")

## Dubois-Prade: redirect conflict to the union

Dubois-Prade redirects each conflicting product `m1(B) · m2(C)` (with `B ∩ C = ∅`) to the union `B ∪ C` — preserving whatever joint support the two sources actually share.

In [ ]:
out_dp = dubois_prade(doctor_1, doctor_2)
print("Dubois-Prade:", out_dp)
print(f"  m({{meningitis, contusion}}) = {out_dp.mass(('meningitis', 'contusion')):.6f}")
print(f"  m(brain_tumor)               = {out_dp.mass(('brain_tumor',)):.6f}")

Most of the joint mass lands on `{meningitis, contusion}`: "the truth is one of those two; we cannot tell which". This is faithful to what the sources actually said.

## PCR5: redistribute conflict proportionally

PCR5 takes each conflicting product `m1(B) · m2(C)` and *splits it back* between `B` and `C` proportionally to the sources' commitments. The brain-tumour stays at a few permille — *not* certainty.

In [ ]:
out_pcr5 = pcr5(doctor_1, doctor_2)
print("PCR5:", out_pcr5)
print(f"  m(meningitis)  = {out_pcr5.mass(('meningitis',)):.6f}")
print(f"  m(contusion)   = {out_pcr5.mass(('contusion',)):.6f}")
print(f"  m(brain_tumor) = {out_pcr5.mass(('brain_tumor',)):.6f}")

Both leading hypotheses hold roughly half the mass, mirroring the doctors' near-symmetric commitments. The brain-tumour stays in the noise. Decision: defer / order more tests / consult a third opinion.

## A side-by-side summary

In [ ]:
rules = [
    ("Dempster",     dempster),
    ("Yager",        yager),
    ("Dubois-Prade", dubois_prade),
    ("PCR5",         pcr5),
    ("PCR6",         pcr6),
    ("Mean",         mean),
]

print(f"{'rule':<14} {'meningitis':>12} {'contusion':>12} {'brain':>10} {'{m,c}':>10} {'Theta':>10}")
for name, rule in rules:
    out = rule(doctor_1, doctor_2)
    print(
        f"{name:<14} "
        f"{out.mass(('meningitis',)):>12.6f} "
        f"{out.mass(('contusion',)):>12.6f} "
        f"{out.mass(('brain_tumor',)):>10.6f} "
        f"{out.mass(('meningitis', 'contusion')):>10.6f} "
        f"{out.mass(theta.omega):>10.6f}"
    )

## A three-source case (PCR5 vs PCR6)

Per `CLAUDE.md` (§"Domain knowledge — pièges critiques", point 3), iterating PCR5 over three or more sources is a methodological error — PCR5 is not associative. The library therefore refuses it and points to PCR6 (Martin-Osswald 2006) for multi-source fusion.

In [ ]:
doctor_3 = MassFunction(theta, {("meningitis",): 0.5, ("contusion",): 0.5})

try:
    pcr5([doctor_1, doctor_2, doctor_3])
except NotImplementedError as exc:
    print("PCR5 refused:", exc)

out_pcr6_3src = pcr6([doctor_1, doctor_2, doctor_3])
print("\nPCR6 (3 sources):", out_pcr6_3src)

## Takeaways

1. **Dempster** assumes the two sources are *independent and individually reliable*. Under high conflict, that assumption is false and the rule produces nonsense. Use it only when conflict is small.
2. **Yager** is conservative: it acknowledges conflict by sending mass to $\Theta$. Useful when you'd rather defer than commit.
3. **Dubois-Prade** keeps the joint support. A reasonable middle ground when most conflict is local (`B ∪ C ≠ Theta`).
4. **PCR5/PCR6** redistribute conflict back to the actual sources' hypotheses. Best for decision-making under high conflict.
5. **Mean** is a simple baseline. Idempotent; never amplifies; never sharpens.

Pick deliberately. Document the choice in your fusion pipeline. And never paper over `K = 1` — `carla-evidence` deliberately raises `TotalConflictError` rather than returning `NaN`.